## Baseline Transformer Inference (DistilBERT)

### Model + Corpus Setup

In [1]:
# Load pre-trained DistilBERT sentiment model (fine-tuned on SST-2)
# We use this as our baseline before adding ESG-specific snippet features.

from transformers import pipeline, AutoTokenizer
import json
import pandas as pd
from pathlib import Path

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment_model = pipeline("sentiment-analysis", model=model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

corpus_path = Path("../data/cleaned_v2/preprocessed_corpus.jsonl")
rows = [json.loads(line) for line in open(corpus_path, encoding="utf-8")]

print(f"✅ Corpus loaded: {len(rows)} documents")


Device set to use mps:0


✅ Corpus loaded: 9 documents


### Chunking Function

In [2]:
# Function to handle long sentences
# If tokenized sentence >512 tokens, split into 510-token chunks
# Average scores from chunks to avoid truncation bias

MAX_CHUNK_TOKENS = 510  # reserve for CLS + SEP

def get_sentiment(text, max_len=MAX_CHUNK_TOKENS):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_len:
        return sentiment_model(text, truncation=True, max_length=512)[0]
    else:
        chunks = [tokens[i:i+max_len] for i in range(0, len(tokens), max_len)]
        scores = []
        for chunk in chunks:
            decoded = tokenizer.decode(chunk, skip_special_tokens=True)
            pred = sentiment_model(decoded, truncation=True, max_length=512)[0]
            pos_score = pred["score"] if pred["label"] == "POSITIVE" else 1 - pred["score"]
            scores.append(pos_score)
        avg_score = sum(scores) / len(scores)
        label = "POSITIVE" if avg_score >= 0.5 else "NEGATIVE"
        return {"label": label, "score": avg_score}

### Apply + Save Results

In [4]:
import pandas as pd

# Load baseline CSV instead of using "results"
df = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

# --- 5-Class Sentiment Mapping ---
def map_to_5class(label, score):
    if 0.45 <= score <= 0.55:
        return "NEUTRAL"
    if label == "NEGATIVE":
        return "VERY NEGATIVE" if score < 0.30 else "NEGATIVE"
    if label == "POSITIVE":
        return "VERY POSITIVE" if score > 0.85 else "POSITIVE"

# Add new column
df["sentiment_5class"] = df.apply(lambda row: map_to_5class(row["label"], row["score"]), axis=1)

# Quick preview
print("✅ Added 5-class sentiment labels")
print(df[["sentence", "label", "score", "sentiment_5class"]].head(10))

# Save to CSV
out_path = "../data/processed/distilbert_baseline_5class.csv"
df.to_csv(out_path, index=False)
print(f"✅ Saved baseline predictions with 5-class labels to {out_path} | rows: {len(df)}")

✅ Added 5-class sentiment labels
                                            sentence     label     score  \
0  environmental report table of contents 1 about...  POSITIVE  0.902096   
1      it also mentions notable targets set in 2022.  POSITIVE  0.972656   
2  this report outlines how we're driving positiv...  POSITIVE  0.999263   
3  for more information about our sustainability ...  POSITIVE  0.615628   
4  for more information about our overall corpora...  POSITIVE  0.914413   
5  for more information about our business, see t...  NEGATIVE  0.929110   
6                 google environmental report 2022 1  NEGATIVE  0.929742   
7  our approach we believe that every business ha...  POSITIVE  0.995205   
8  sustainability is one of our core values at go...  POSITIVE  0.999007   
9  we've been a leader on sustainability and clim...  POSITIVE  0.999525   

  sentiment_5class  
0    VERY POSITIVE  
1    VERY POSITIVE  
2    VERY POSITIVE  
3         POSITIVE  
4    VERY POSITIVE  
5   